In [1]:
import lancedb
from lancedb.pydantic import LanceModel, Vector, pydantic_to_schema
from pydantic import PlainSerializer, TypeAdapter, Field, BeforeValidator
from typing import Annotated
import uuid
from enum import Enum
from datetime import datetime
import pyarrow as pa
from typing import Optional
# import geneva
# from geneva import udf

## Schemas

In [66]:
clip_embedding_dim = 3
tower_embedding_dim = 2

class Cover(LanceModel):
    cover_id: int
    book_id: int
    isbn_13: str
    cover_url: str
    cover_embedding: Vector(clip_embedding_dim) #pyright: ignore[reportInvalidTypeForm]
    tower_embedding: Optional[Vector(tower_embedding_dim)] = None #pyright: ignore[reportInvalidTypeForm]

class User(LanceModel):
    user_id: Annotated[uuid.UUID, PlainSerializer(lambda x: x.bytes, return_type=bytes)]
    tower_embedding: Vector(tower_embedding_dim) #pyright: ignore[reportInvalidTypeForm]

class InteractionEnum(str, Enum):
    rating = 'rating'

class Interaction(LanceModel):
    user_id: Annotated[uuid.UUID, PlainSerializer(lambda x: x.bytes, return_type=bytes)]
    cover_id: int
    type: InteractionEnum
    score: int
    timestamp: datetime

class RunlogEnum(str, Enum):
    fine_tuning = 'fine_tuning'

class Runlog(LanceModel):
    type: RunlogEnum
    last_run: datetime

In [3]:
clip_dim = 3
tower_dim = 2

cover_schema = pa.schema(
    [
        pa.field("cover_id", pa.int64(), nullable=False),
        pa.field("book_id", pa.int64(), nullable=False),
        pa.field("isbn_13", pa.string(), nullable=False),
        pa.field("cover_url", pa.string(), nullable=False),
        pa.field("cover_embedding", pa.fixed_shape_tensor(pa.float32(), (clip_dim,)), nullable=False),
        pa.field("tower_embedding", pa.list_(pa.float32(), tower_dim), nullable=True),
    ]
)

user_schema = pa.schema(
    [
        pa.field("user_id", pa.uuid(), nullable=False),
        pa.field("tower_embedding", pa.list_(pa.float32(), tower_dim), nullable=False),
    ]
)

interaction_schema = pa.schema(
    [
        pa.field("user_id", pa.uuid(), nullable=False),
        pa.field("cover_id", pa.int64(), nullable=False),
        pa.field("type", pa.string(), nullable=False),
        pa.field("score", pa.int64(), nullable=False),
        pa.field("timestamp", pa.timestamp('us'), nullable=False),
    ]
)

runlog_schema = pa.schema(
    [
        pa.field("type", pa.string(), nullable=False),
        pa.field("last_run", pa.timestamp('us'), nullable=False),
    ]
)

In [4]:
interaction_schema

user_id: extension<arrow.uuid> not null
cover_id: int64 not null
type: string not null
score: int64 not null
timestamp: timestamp[us] not null

In [89]:
User.to_arrow_schema()

TypeError: Converting Pydantic type to Arrow Type: unsupported type <class 'uuid.UUID'>.

In [5]:
import pyarrow as pa

user = User(user_id=uuid.uuid4(), tower_embedding=[0, 1])
data = [user.model_dump()]
table = pa.Table.from_pylist(data)
print(table)

pyarrow.Table
user_id: binary
tower_embedding: list<item: double>
  child 0, item: double
----
user_id: [[EE4013313A744192AC297588AB08CC47]]
tower_embedding: [[[0,1]]]


In [5]:
uri = "test_lancedb"
#db = geneva.connect(uri)
db = lancedb.connect(uri)

In [6]:
cover_table = db.create_table(
    "covers", schema=cover_schema, exist_ok=True
)
user_table = db.create_table(
    "users", schema=user_schema, exist_ok=True
)
interaction_table = db.create_table(
    "interactions", schema=interaction_schema, exist_ok=True
)
runlog_table = db.create_table(
    "runlog", schema=runlog_schema, exist_ok=True
)

In [7]:
id_stats = cover_table.index_stats("cover_id_idx")
if not id_stats:
    cover_table.create_index("cover_id", config=lancedb.index.BTree(), name="cover_id_idx")

id_stats = user_table.index_stats("user_id_idx")
if not id_stats:
    user_table.create_index("user_id", config=lancedb.index.BTree(), name="user_id_idx")

user_id_stats = interaction_table.index_stats("user_id_idx")
cover_id_stats = interaction_table.index_stats("cover_id_idx")
if not user_id_stats or not cover_id_stats:
    interaction_table.create_index("user_id", config=lancedb.index.BTree(), name="user_id_idx")
    interaction_table.create_index("cover_id", config=lancedb.index.BTree(), name="cover_id_idx")

type_stats = runlog_table.index_stats("type_idx")
if not type_stats:
    runlog_table.create_index("type", config=lancedb.index.BTree(), name="type_idx")

In [8]:
interaction_table.list_indices()

[IndexConfig(name="cover_id_idx", index_type="BTree", columns=["cover_id"], index_uuid="48c18756-7a13-4749-ad81-7366e59c5183", type_url="/lance.table.BTreeIndexDetails", created_at=datetime.datetime(2026, 7, 24, 9, 10, 44, 488000, tzinfo=datetime.timezone.utc), num_indexed_rows=0, num_unindexed_rows=0, size_bytes=713, num_segments=1, index_version=0, index_details={}),
 IndexConfig(name="user_id_idx", index_type="BTree", columns=["user_id"], index_uuid="2c3a67af-9694-4edf-a174-bbf9001b0d7a", type_url="/lance.table.BTreeIndexDetails", created_at=datetime.datetime(2026, 7, 24, 9, 10, 44, 488000, tzinfo=datetime.timezone.utc), num_indexed_rows=0, num_unindexed_rows=0, size_bytes=758, num_segments=1, index_version=0, index_details={})]

## Ingesting Data

In [16]:
covers_adapter = TypeAdapter(list[Cover])

cover_list = [
    Cover(cover_id=1, book_id=2, isbn_13="1234567891011", cover_url="something.cool.com/bruh.jpg", cover_embedding=[1, 3, 9]),
    Cover(cover_id=5, book_id=2, isbn_13="1234567891014", cover_url="something.cool.com/bruh2.jpg", cover_embedding=[15, 2, 2], tower_embedding=[3, 2])
]

In [11]:
temp1 = [
    Cover(cover_id=1, book_id=2, isbn_13="1234567891011", cover_url="something.cool.com/bruh.jpg", cover_embedding=[1, 3, 9]),
    Cover(cover_id=5, book_id=2, isbn_13="1234567891014", cover_url="something.cool.com/bruh2.jpg", cover_embedding=[15, 2, 2], tower_embedding=[3, 2])
]
temp2 = [
    Cover(cover_id=1, book_id=2, isbn_13="1234567891011", cover_url="something.cool.com/bruh.jpg", cover_embedding=[1, 3, 9]),
    Cover(cover_id=8, book_id=2, isbn_13="1234567891015", cover_url="something.cool.com/bruh3.jpg", cover_embedding=[5, 12, 0])
]

together_map = {item.cover_id: item for item in temp1 + temp2}
print(list(together_map.keys()))
list(together_map.values())

[1, 5, 8]


[Cover(cover_id=1, book_id=2, isbn_13='1234567891011', cover_url='something.cool.com/bruh.jpg', cover_embedding=FixedSizeList(dim=3), tower_embedding=None),
 Cover(cover_id=5, book_id=2, isbn_13='1234567891014', cover_url='something.cool.com/bruh2.jpg', cover_embedding=FixedSizeList(dim=3), tower_embedding=FixedSizeList(dim=2)),
 Cover(cover_id=8, book_id=2, isbn_13='1234567891015', cover_url='something.cool.com/bruh3.jpg', cover_embedding=FixedSizeList(dim=3), tower_embedding=None)]

In [17]:
(
    cover_table.merge_insert("cover_id")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute(covers_adapter.dump_python(cover_list))
)

MergeResult(version=5, num_updated_rows=2, num_inserted_rows=0, num_deleted_rows=0, num_attempts=1, num_rows=2)

In [13]:
cover_table.head()

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: extension<arrow.fixed_shape_tensor[value_type=float, shape=[3]]> not null
tower_embedding: fixed_size_list<item: float>[2]
  child 0, item: float
----
cover_id: [[1,5]]
book_id: [[2,2]]
isbn_13: [["1234567891011","1234567891014"]]
cover_url: [["something.cool.com/bruh.jpg","something.cool.com/bruh2.jpg"]]
cover_embedding: [[[1,3,9],[15,2,2]]]
tower_embedding: [[null,[3,2]]]

In [77]:
import torch
from lancedb.permutation import Permutation, permutation_builder

class PopularCoversDataSet(torch.utils.data.Dataset):
    def __init__(self, 
        table: lancedb.Table, cover_ids: list[int], cover_id_field: str = "cover_id", 
        embedding_field: str = "cover_embedding", max_rating: float = 5.0
    ):
        self.table = table
        self.cover_ids = cover_ids
        self.cover_id_field = cover_id_field
        self.embedding_field = embedding_field
        self.max_rating = max_rating
        self.default_user_id = uuid.UUID(int=0)
        self.default_user = self.get_default_user()
        self.rating_arr = torch.tensor([self.max_rating])
        self.perm: Permutation | None = None
    
    def __len__(self):
        return len(self.cover_ids)

    def get_default_user(self) -> torch.Tensor:
        return torch.frombuffer(self.default_user_id.bytes_le, dtype=torch.int32).to(dtype=torch.float32).unsqueeze(0)
    
    def _ensure_permutation(self) -> Permutation:
        if self.perm is None:
            id_strings = [f'{cid}' for cid in self.cover_ids]
            permutation_tbl = (
                permutation_builder(self.table)
                .filter(f"{self.cover_id_field} IN ({', '.join(id_strings)})")
                .execute()
            )
            permutation = (
                Permutation.from_tables(self.table, permutation_tbl)
                .select_columns(["cover_id", "cover_embedding"])
            )
            self.perm = permutation

    def __getitem__(self, idx: int):
        self._ensure_permutation()
        cover = self.perm.__getitem__(idx)[0]
        item_arr = torch.tensor(cover[self.embedding_field])
        rating_arr = torch.tensor([self.max_rating])
        
        return self.default_user, item_arr, rating_arr

In [23]:
dataset = PopularCoversDataSet(cover_table, cover_ids=[5, 1])

dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

for batch in dataloader:
    print(batch)

[tensor([[[0., 0., 0., 0.]]]), tensor([[1., 3., 9.]]), tensor([[5.]])]
[tensor([[[0., 0., 0., 0.]]]), tensor([[15.,  2.,  2.]]), tensor([[5.]])]


In [121]:
import torch
from lancedb.util import tbl_to_tensor

dataloader = torch.utils.data.DataLoader(
    cover_table, batch_size=1, shuffle=True
)

for batch in dataloader:
    print(batch)

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'pyarrow.lib.ChunkedArray'>

In [103]:
import torch
from lancedb.permutation import Permutation, permutation_builder

permutation = Permutation.identity(cover_table).select_columns(["cover_id", "cover_embedding"])
print(permutation.__getitem__(0))
dataloader = torch.utils.data.DataLoader(
    permutation, batch_size=1, shuffle=True
)

for batch in dataloader:
    print(batch)

[{'cover_id': 5, 'cover_embedding': [15.0, 2.0, 2.0]}]
{'cover_id': tensor([1]), 'cover_embedding': [tensor([1.], dtype=torch.float64), tensor([3.], dtype=torch.float64), tensor([9.], dtype=torch.float64)]}
{'cover_id': tensor([5]), 'cover_embedding': [tensor([15.], dtype=torch.float64), tensor([2.], dtype=torch.float64), tensor([2.], dtype=torch.float64)]}


In [ ]:
stuff = {'cover_id': 5, 'cover_embedding': [15.0, 2.0, 2.0]}


In [ ]:
test_list = [1]
[f'{id}' for id in test_list]

TypeError: sequence item 0: expected str instance, int found

In [78]:
test_list = [1]
permutation_tbl = permutation_builder(cover_table).filter(f"cover_id IN ({', '.join([f'{id}' for id in test_list])})").execute()
permutation = Permutation.from_tables(cover_table, permutation_tbl).select_columns(["cover_id", "cover_embedding"])

dataloader = torch.utils.data.DataLoader(
    permutation, batch_size=1, shuffle=True
)

for batch in dataloader:
    print(batch)

{'cover_id': tensor([1]), 'cover_embedding': [tensor([1.], dtype=torch.float64), tensor([3.], dtype=torch.float64), tensor([9.], dtype=torch.float64)]}


In [25]:
users_adapter = TypeAdapter(list[User])

user_list = [
    User(user_id=uuid.uuid4(), tower_embedding=[1, 3]),
]

In [32]:
(
    user_table.merge_insert("user_id")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute(users_adapter.dump_python(user_list))
)

MergeResult(version=7, num_updated_rows=0, num_inserted_rows=1, num_deleted_rows=0, num_attempts=1, num_rows=1)

In [33]:
user_table.head()

pyarrow.Table
user_id: extension<arrow.uuid> not null
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
----
user_id: [[6A25CE1D84134CC8BD223795E1C7E4AB,A509DB1416E14250A32CCC752F7B23DA,00000000000000000000000000000000],[CE74F24A751445AD9499658D2FE6329F],[2925670C0E6C4D419EE3186D3177A7A8]]
tower_embedding: [[[1780862500,1288209400],[-1526080800,1112545000],[2,2]],[[1,3]],[[1,3]]]

In [26]:
interactions_adapter = TypeAdapter(list[Interaction])

interaction_list = [
    Interaction(user_id=user_list[0].user_id, cover_id=cover_list[1].cover_id, type=InteractionEnum.rating, score=4, timestamp=datetime.now())
]

In [27]:
interaction_table.add(interactions_adapter.dump_python(interaction_list))

AddResult(version=6)

In [28]:
interaction_table.head()

pyarrow.Table
user_id: extension<arrow.uuid> not null
cover_id: int64 not null
type: string not null
score: int64 not null
timestamp: timestamp[us] not null
----
user_id: [[CE74F24A751445AD9499658D2FE6329F],[CE74F24A751445AD9499658D2FE6329F],[2925670C0E6C4D419EE3186D3177A7A8]]
cover_id: [[5],[1],[5]]
type: [["rating"],["rating"],["rating"]]
score: [[4],[2],[4]]
timestamp: [[2026-07-29 00:04:41.460083],[2026-07-29 00:05:20.022649],[2026-07-29 00:05:55.524392]]

In [69]:
interactions_res = (
    interaction_table.search()
    .select(["user_id", "cover_id", "type", "score", "timestamp"])
).to_pydantic(Interaction)

interactions_res

[Interaction(user_id=UUID('ce74f24a-7514-45ad-9499-658d2fe6329f'), cover_id=5, type=<InteractionEnum.rating: 'rating'>, score=4, timestamp=datetime.datetime(2026, 7, 29, 0, 4, 41, 460083)),
 Interaction(user_id=UUID('ce74f24a-7514-45ad-9499-658d2fe6329f'), cover_id=1, type=<InteractionEnum.rating: 'rating'>, score=2, timestamp=datetime.datetime(2026, 7, 29, 0, 5, 20, 22649)),
 Interaction(user_id=UUID('2925670c-0e6c-4d41-9ee3-186d3177a7a8'), cover_id=5, type=<InteractionEnum.rating: 'rating'>, score=4, timestamp=datetime.datetime(2026, 7, 29, 0, 5, 55, 524392))]

In [101]:
class FeedbackMap(tuple[int, int], Enum):
    Rating = (0, 3)
    other_thing = (0, 3)

FeedbackMap.Rating

<FeedbackMap.Rating: (0, 3)>

In [106]:
FeedbackMap["Rating"]

<FeedbackMap.Rating: (0, 3)>

In [ ]:
FeedbackMap.Rating.value

(0, 3)

In [91]:
FeedbackMap.rating.other_thing

<FeedbackMap.rating: (0, 3)>

In [85]:
interactions_res[0].score

4

In [63]:
len(interactions_res)

3

In [64]:
interactions_res[0]

{'user_id': UUID('ce74f24a-7514-45ad-9499-658d2fe6329f'),
 'cover_id': 5,
 'type': 'rating',
 'score': 4,
 'timestamp': datetime.datetime(2026, 7, 29, 0, 4, 41, 460083)}

In [58]:
cover_ids = interactions_res.column("cover_id").to_pylist()
cover_ids

[5, 1, 5]

In [55]:
covers_res = (
    cover_table.search()
    .where(f"cover_id IN ({', '.join([f'{id}' for id in cover_ids])})")
    .select(["cover_id", "cover_embedding"])
    .limit(len(cover_ids))
).to_polars()

covers_res

cover_id,cover_embedding
i64,"array[f32, 3]"
1,"[1.0, 3.0, 9.0]"
5,"[15.0, 2.0, 2.0]"


In [81]:
class CoverBackdate(LanceModel):
    cover_id: int
    cover_embedding: Vector(clip_embedding_dim)  # pyright: ignore[reportInvalidTypeForm]

covers_res_one = (
    cover_table.search()
    .where(f"cover_id = {cover_ids[0]}")
    .select(["cover_id", "cover_embedding"])
    .limit(1)
).to_pydantic(CoverBackdate)

covers_res_one

[CoverBackdate(cover_id=5, cover_embedding=FixedSizeList(dim=3))]

In [ ]:
def process_user_id(user_id: uuid.UUID) -> torch.Tensor:
    bytes_copy = bytearray(user_id.bytes_le)
    return (
        torch.frombuffer(bytes_copy, dtype=torch.int32)
        .to(dtype=torch.float32).unsqueeze(0)
    )

class FeedbackMap(tuple[int, int], Enum):
    rating = (0, 3)

class FeedbackDataSet(torch.utils.data.Dataset):
    def __init__(
            self, feedback_table: lancedb.Table, cover_table: lancedb.Table,
            uid_field: str = "user_id", cid_field: str = "cover_id",
            embedding_field: str = "cover_embedding"
        ):
        self.feedback_table = feedback_table
        self.cover_table = cover_table
        self.uid_field = uid_field
        self.cid_field = cid_field
        self.embedding_field = embedding_field
        self.feedback_list = self._load_feedback_list()

    def __len__(self):
        return len(self.feedback_list)

    def _load_feedback_list(self) -> list[Interaction]:
        return (
            self.feedback_table.search()
            .select([self.uid_field, self.cid_field, "type", "score", "timestamp"])
        ).to_pydantic(Interaction)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        feedback = self.feedback_list[idx]
        cover = (
            self.cover_table.search()
            .where(f"{self.cid_field} = {feedback.cover_id}")
            .select([self.cid_field, self.embedding_field])
            .limit(1)
        ).to_pydantic(CoverBackdate)[0]

        user_arr = process_user_id(feedback.user_id)
        item_arr = torch.tensor(cover.cover_embedding)
        rating_arr = torch.tensor([feedback.score])
        min_rating_arr = torch.tensor([FeedbackMap[feedback.type.value].value[0]])
        max_rating_arr = torch.tensor([FeedbackMap[feedback.type.value].value[1]])

        return user_arr, item_arr, rating_arr, min_rating_arr, max_rating_arr

In [113]:
dataset = FeedbackDataSet(interaction_table, cover_table)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

for batch in dataloader:
    print(batch)

[tensor([[[-8.3120e+08,  1.1690e+09, -1.9227e+09, -1.6241e+09]]]), tensor([[15.,  2.,  2.]]), tensor([[4]]), tensor([[0]]), tensor([[3]])]
[tensor([[[-8.3120e+08,  1.1690e+09, -1.9227e+09, -1.6241e+09]]]), tensor([[1., 3., 9.]]), tensor([[2]]), tensor([[0]]), tensor([[3]])]
[tensor([[[ 6.9032e+08,  1.2961e+09,  1.8303e+09, -1.4654e+09]]]), tensor([[15.,  2.,  2.]]), tensor([[4]]), tensor([[0]]), tensor([[3]])]


In [82]:
covers_res_one[0].cover_embedding

FixedSizeList(dim=3)

In [83]:
torch.tensor(covers_res_one[0].cover_embedding)

tensor([15.,  2.,  2.])

In [47]:
interactions_res.join(covers_res, 'cover_id')

user_id,cover_id,type,score,timestamp,cover_embedding
binary,i64,str,i64,datetime[μs],"array[f32, 3]"
"b""\xcet\xf2Ju\x14E\xad\x94\x99e\x8d/\xe62\x9f""",5,"""rating""",4,2026-07-29 00:04:41.460083,"[15.0, 2.0, 2.0]"
"b""\xcet\xf2Ju\x14E\xad\x94\x99e\x8d/\xe62\x9f""",1,"""rating""",2,2026-07-29 00:05:20.022649,"[1.0, 3.0, 9.0]"
"b"")%g\x0c\x0elMA\x9e\xe3\x18m1w\xa7\xa8""",5,"""rating""",4,2026-07-29 00:05:55.524392,"[15.0, 2.0, 2.0]"


In [60]:
runlog_adapter = TypeAdapter(list[Runlog])

runlog = [Runlog(type=RunlogEnum.fine_tuning, last_run=datetime.now())]

In [61]:
(
    runlog_table.merge_insert("type")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute(runlog_adapter.dump_python(runlog))
)

MergeResult(version=3, num_updated_rows=0, num_inserted_rows=1, num_deleted_rows=0, num_attempts=1, num_rows=1)

In [62]:
runlog_table.head()

pyarrow.Table
type: string not null
last_run: timestamp[us] not null
----
type: [["fine_tuning"]]
last_run: [[2026-07-17 18:37:31.123428]]

## Querying

In [63]:
current_user_id = user_list[0].user_id.hex
current_user_id

'9eb87f4c539140e2a1045f86c45d3532'

In [64]:
embedding = (
    user_table.search()
    .where(f"user_id = X'{current_user_id}'")
    .select(["tower_embedding"])
    .limit(1)
    .to_list()
)[0]["tower_embedding"]
embedding

[1.0, 3.0]

In [65]:
covers = (
    cover_table.search(embedding, vector_column_name="tower_embedding")
    .select(["cover_id", "book_id", "isbn_13", "cover_url", "tower_embedding", "_distance"])
    .limit(10)
    .to_list()
)
covers

[{'cover_id': 1,
  'book_id': 2,
  'isbn_13': '1234567891011',
  'cover_url': 'something.cool.com/bruh.jpg',
  'tower_embedding': [1.0, 2.0],
  '_distance': 1.0},
 {'cover_id': 5,
  'book_id': 2,
  'isbn_13': '1234567891014',
  'cover_url': 'something.cool.com/bruh2.jpg',
  'tower_embedding': [3.0, 2.0],
  '_distance': 5.0}]

In [66]:
cover_table.head()

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: fixed_size_list<item: float>[3] not null
  child 0, item: float
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
----
cover_id: [[5,1]]
book_id: [[2,2]]
isbn_13: [["1234567891014","1234567891011"]]
cover_url: [["something.cool.com/bruh2.jpg","something.cool.com/bruh.jpg"]]
cover_embedding: [[[15,2,2],[1,3,9]]]
tower_embedding: [[[3,2],[1,2]]]

In [27]:
all_ids = (
    user_table.search()
    .select(["user_id"])
).to_list()

user_id_list = {uuid.UUID(int=0)}
for user in all_ids:
    user_id_list.add(user["user_id"])
    print(user["user_id"].bytes_le)

b'\x14\xdb\t\xa5\xe1\x16PB\xa3,\xccu/{#\xda'
b'\x1d\xce%j\x13\x84\xc8L\xbd"7\x95\xe1\xc7\xe4\xab'


In [33]:
all_ids[0]

{'user_id': UUID('a509db14-16e1-4250-a32c-cc752f7b23da')}

In [35]:
for user in all_ids:
    if user["user_id"] == all_ids[0]["user_id"]:
        print("Bruh")
        break;
    user_id_list.add(user["user_id"])
    print(user["user_id"].bytes_le)

Bruh


In [57]:
import torch

def process_user_id(user_id: uuid.UUID) -> torch.Tensor:
    bytes_copy = bytearray(user_id.bytes_le)
    return (
        torch.frombuffer(bytes_copy, dtype=torch.int32)
        .to(dtype=torch.float32).unsqueeze(0)
    ) + 2

tensors = torch.vstack([process_user_id(uid) for uid in user_id_list])
tensors

tensor([[ 2.0000e+00,  2.0000e+00,  2.0000e+00,  2.0000e+00],
        [ 1.7809e+09,  1.2882e+09, -1.7915e+09, -1.4111e+09],
        [-1.5261e+09,  1.1125e+09,  1.9763e+09, -6.3521e+08]])

In [58]:
tensors.shape

torch.Size([3, 4])

In [59]:
tensors_list = torch.unbind(tensors, dim=0)
tensors_list

(tensor([2., 2., 2., 2.]),
 tensor([ 1.7809e+09,  1.2882e+09, -1.7915e+09, -1.4111e+09]),
 tensor([-1.5261e+09,  1.1125e+09,  1.9763e+09, -6.3521e+08]))

In [53]:
togetha = [User(user_id=uid, tower_embedding=tensor[:2]) for uid, tensor in zip(user_id_list, tensors_list)]
togetha

[User(user_id=UUID('00000000-0000-0000-0000-000000000000'), tower_embedding=FixedSizeList(dim=2)),
 User(user_id=UUID('6a25ce1d-8413-4cc8-bd22-3795e1c7e4ab'), tower_embedding=FixedSizeList(dim=2)),
 User(user_id=UUID('a509db14-16e1-4250-a32c-cc752f7b23da'), tower_embedding=FixedSizeList(dim=2))]

In [54]:
users_adapter.dump_python(togetha)

[{'user_id': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
  'tower_embedding': [2.0, 2.0]},
 {'user_id': b'j%\xce\x1d\x84\x13L\xc8\xbd"7\x95\xe1\xc7\xe4\xab',
  'tower_embedding': [1780862464.0, 1288209408.0]},
 {'user_id': b'\xa5\t\xdb\x14\x16\xe1BP\xa3,\xccu/{#\xda',
  'tower_embedding': [-1526080768.0, 1112545024.0]}]

In [55]:
(
    user_table.merge_insert("user_id")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute(users_adapter.dump_python(togetha))
)

MergeResult(version=5, num_updated_rows=2, num_inserted_rows=1, num_deleted_rows=0, num_attempts=1, num_rows=3)

In [56]:
user_table.head()

pyarrow.Table
user_id: extension<arrow.uuid> not null
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
----
user_id: [[6A25CE1D84134CC8BD223795E1C7E4AB,A509DB1416E14250A32CCC752F7B23DA,00000000000000000000000000000000]]
tower_embedding: [[[1780862500,1288209400],[-1526080800,1112545000],[2,2]]]

## Adding Embeddings

In [67]:
import torch

In [ ]:
cover_table.add_columns({"random_embedding": f"arrow_cast(NULL, 'FixedSizeList({tower_embedding_dim}, Float32)')"})

AddColumnsResult(version=6)

In [60]:
cover_table.head()

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: extension<arrow.fixed_shape_tensor[value_type=float, shape=[3]]> not null
tower_embedding: fixed_size_list<item: float>[2]
  child 0, item: float
----
cover_id: [[1,5]]
book_id: [[2,2]]
isbn_13: [["1234567891011","1234567891014"]]
cover_url: [["something.cool.com/bruh.jpg","something.cool.com/bruh2.jpg"]]
cover_embedding: [[[1,3,9],[15,2,2]]]
tower_embedding: [[null,[3,2]]]

In [94]:
cover_ids = []
cover_embeddings = []
db_cover_dict = cover_table.search().select(["cover_id", "cover_embedding"]).to_list()

db_cover_dict

[{'cover_id': 1, 'cover_embedding': [1.0, 3.0, 9.0]},
 {'cover_id': 5, 'cover_embedding': [15.0, 2.0, 2.0]}]

In [95]:
for cover in db_cover_dict:
    cover_ids.append(cover["cover_id"])
    cover_embeddings.append(cover["cover_embedding"])

print(cover_ids)
cover_embeddings

[1, 5]


[[1.0, 3.0, 9.0], [15.0, 2.0, 2.0]]

In [62]:
cover_embeddings

[[1.0, 3.0, 9.0], [15.0, 2.0, 2.0]]

In [70]:
cover_tensors = torch.vstack([torch.tensor(embed) for embed in cover_embeddings])
cover_tensors

tensor([[ 1.,  3.,  9.],
        [15.,  2.,  2.]])

In [71]:
cover_tensors.shape

torch.Size([2, 3])

In [99]:
class CoverUpdate(LanceModel):
    cover_id: int
    tower_embedding: Vector(tower_embedding_dim) #pyright: ignore[reportInvalidTypeForm]

tower_embedding_list = torch.unbind(cover_tensors, dim=0)
cover_update_list = [
    CoverUpdate(cover_id=cid, tower_embedding=tensor[:2])
    for cid, tensor in zip(cover_ids, tower_embedding_list)
]
cover_update_list

[CoverUpdate(cover_id=1, tower_embedding=FixedSizeList(dim=2)),
 CoverUpdate(cover_id=5, tower_embedding=FixedSizeList(dim=2))]

In [100]:
cover_updates_adapter = TypeAdapter(list[CoverUpdate])
cover_updates_adapter.dump_python(cover_update_list)

[{'cover_id': 1, 'tower_embedding': [1.0, 3.0]},
 {'cover_id': 5, 'tower_embedding': [15.0, 2.0]}]

In [77]:
class CoverUpdate(LanceModel):
    cover_id: int
    random_embedding: Vector(tower_embedding_dim) #pyright: ignore[reportInvalidTypeForm]

cover_updates_adapter = TypeAdapter(list[CoverUpdate])

cover_update_list = [
    CoverUpdate(cover_id=1, random_embedding=[4, 3]),
    CoverUpdate(cover_id=5, random_embedding=[6, 1]),
]

In [78]:
(
    cover_table.merge_insert("cover_id")
    .when_matched_update_all()
    .execute(cover_updates_adapter.dump_python(cover_update_list))
)

MergeResult(version=7, num_updated_rows=2, num_inserted_rows=0, num_deleted_rows=0, num_attempts=1, num_rows=2)

In [79]:
cover_table.head()

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: fixed_size_list<item: float>[3] not null
  child 0, item: float
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
random_embedding: fixed_size_list<item: float>[2]
  child 0, item: float
----
cover_id: [[1,5]]
book_id: [[2,2]]
isbn_13: [["1234567891011","1234567891014"]]
cover_url: [["something.cool.com/bruh.jpg","something.cool.com/bruh2.jpg"]]
cover_embedding: [[[1,3,9],[15,2,2]]]
tower_embedding: [[[1,2],[3,2]]]
random_embedding: [[[4,3],[6,1]]]